In [1]:
!pip install -U langchain langchain-core langchain-openai
!pip install -U langchain-community

In [9]:
# Imports
from langchain_openai import ChatOpenAI
from langchain_classic.agents import create_react_agent
from langchain_classic.agents.agent import AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from typing import List

In [10]:
# Define Custom Tools (Text Processor and Calculator)

@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression string (e.g., '2 + 3 * 5'). Use this for all math problems."""
    try:
        # NOTE: Using eval() is insecure in production, use safer alternatives like numexpr or a dedicated calculator chain/library.
        return str(eval(expression))
    except Exception as e:
        return f"Error evaluating expression: {e}"

@tool
def text_processor(text_to_summarize: str) -> str:
    """A tool to summarize a long text string, like a web search result, into its key points."""
    # In a real implementation, this would call the LLM again or use a dedicated summarization model/library.
    # For this sketch, we will use a simple placeholder or a small LLM call.
    # --- Simplified Stub ---
    return f"Summarized key points: The text is about X, Y, and Z. Length: {len(text_to_summarize)}"

In [17]:
from google.colab import userdata
TAVILY_API_KEY=userdata.get('TAVILY_API_KEY')
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')

In [14]:
# Define External Tools (Web Search)
web_search = TavilySearchResults(k=3, tavily_api_key=TAVILY_API_KEY)
web_search.name = "internet_search"
web_search.description = "A tool to search the internet for current events, facts, or general knowledge."

In [15]:
# Create Tool List
tools = [calculator, text_processor, web_search]

In [45]:
# 5. Define LLM and Prompt
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY) # Use a good model for reasoning

system_prompt = (
    "You are an expert autonomous assistant. Your goal is to break down complex queries, "
    "determine the necessary sequence of tool calls (search, calculate, process), and "
    "provide a final, well-structured answer to the user. Always use your tools when necessary.\n\n"
    "Use the following format:\n\n"
    "Question: the input question you must answer\n"
    "Thought: you should always think about what to do\n"
    "Action: the action to take, should be one of [{tool_names}]\n"
    "Action Input: the input to the action\n"
    "Observation: the result of the action\n"
    "... (this Thought/Action/Action Input/Observation can repeat N times)\n"
    "Thought: I now know the final answer\n"
    "Final Answer: the final answer to the original input question\n\n"
    "Begin!\n"
    "Question: {input}\n"
    "Thought:{agent_scratchpad}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", f"{system_prompt}\n\nAvailable tools: {{tools}}\nTool names: {{tool_names}}"),
    ("user", "{input}"),
    ("human", "{agent_scratchpad}")
])

In [46]:
#  Create Agent and Executor
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

In [47]:
user_query = "What is 15% of the current population of the country with the capital 'Wellington'? Also, provide a 5-word summary of the search results you used."
# Run the Agent
result = agent_executor.invoke({"input": user_query})
print("\n--- FINAL ANSWER ---")
print(result["output"])



> Entering new AgentExecutor chain...
Thought: I need to find the current population of the country with the capital 'Wellington', which is New Zealand. After that, I can calculate 15% of that population. I will perform an internet search to find the current population of New Zealand. 
Action: internet_search
Action Input: "current population of New Zealand 2023"[{'title': 'New Zealand Population (2025) - Worldometer', 'url': 'https://www.worldometers.info/world-population/new-zealand-population/', 'content': "The current population of New Zealand is 5,268,340 as of Monday, December 8, 2025, based on Worldometer's elaboration of the latest United Nations data1.", 'score': 0.9997347}, {'title': 'New Zealand Population (1950-2025)', 'url': 'https://www.macrotrends.net/global-metrics/countries/nzl/new-zealand/population', 'content': 'Total population for New Zealand in 2023 was 5,223,100, a 2.07% increase from 2022. Total population for New Zealand in 2022 was 5,117,200, a 0.12% increas